# Laboration 1

## data_loading_code.py

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from matplotlib import pyplot
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk import word_tokenize
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, classification_report
from torch.nn.utils.rnn import pad_sequence
from collections import defaultdict
import nltk # new
nltk.download('stopwords') # new
nltk.download('punkt_tab')
def preprocess_pandas(data, columns):
    df_ = pd.DataFrame(columns=columns)
    data['Sentence'] = data['Sentence'].str.lower()
    data['Sentence'] = data['Sentence'].replace('[a-zA-Z0-9-_.]+@[a-zA-Z0-9-_.]+', '', regex=True)                      # remove emails
    data['Sentence'] = data['Sentence'].replace('((25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)(\.|$)){4}', '', regex=True)    # remove IP address
    data['Sentence'] = data['Sentence'].str.replace(r'[^\w\s]','')                                                       # remove special characters
    data['Sentence'] = data['Sentence'].replace(r'\d', '', regex=True)                                                   # remove numbers
    for index, row in data.iterrows():
        word_tokens = word_tokenize(row['Sentence'])
        filtered_sent = [w for w in word_tokens if not w in stopwords.words('english')]
        df_.loc[len(df_)] = {
            "index": row['index'],
            "Class": row['Class'],
            "Sentence": " ".join(filtered_sent)
        }
    return data

# If this is the primary file that is executed (ie not an import of another file)
# if __name__ == "__main__":
# get data, pre-process and split
data = pd.read_csv("amazon_cells_labelled.txt", delimiter='\t', header=None)
data.columns = ['Sentence', 'Class']
data['index'] = data.index                                          # add new column index
columns = ['index', 'Class', 'Sentence']
data = preprocess_pandas(data, columns)                             # pre-process
training_data, validation_data, training_labels, validation_labels = train_test_split( # split the data into training, validation, and test splits
    data['Sentence'].values.astype('U'),
    data['Class'].values.astype('int32'),
    test_size=0.10,
    random_state=0,
    shuffle=True
)

# vectorize data using TFIDF and transform for PyTorch for scalability
word_vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1,2), max_features=50000, max_df=0.5, use_idf=True, norm='l2')
training_data = word_vectorizer.fit_transform(training_data)        # transform texts to sparse matrix
training_data = training_data.todense()                             # convert to dense matrix for Pytorch
vocab_size = len(word_vectorizer.vocabulary_)
validation_data = word_vectorizer.transform(validation_data)
validation_data = validation_data.todense()
train_x_tensor = torch.from_numpy(np.array(training_data)).type(torch.FloatTensor)
train_y_tensor = torch.from_numpy(np.array(training_labels)).long()
validation_x_tensor = torch.from_numpy(np.array(validation_data)).type(torch.FloatTensor)
validation_y_tensor = torch.from_numpy(np.array(validation_labels)).long()

# def load_data(filepath):
#     data = pd.read_csv(filepath, delimiter='\t', header=None)
#     data.columns = ['Sentence', 'Class']
#     data['index'] = data.index
#     columns = ['index', 'Class', 'Sentence']
#     data = preprocess_pandas(data, columns)
#     training_data, validation_data, training_labels, validation_labels = train_test_split(
#         data['Sentence'].values.astype('U'), # Unicode string
#         data['Class'].values.astype(np.int32),
#         test_size=0.15,
#         shuffle=True
#     )

# # Build vocabulary
#     word_to_idx = defaultdict(lambda: 0)  # 0 will be padding
#     idx = 1
#     for sentence in training_data:
#         for word in sentence.split():
#             if word not in word_to_idx:
#                 word_to_idx[word] = idx
#                 idx += 1
#     vocab_size = len(word_to_idx) + 1  # +1 for padding index 0

#     # Convert sentences to sequences of word indices
#     def sentences_to_indices(sentences, word_to_idx):
#         seqs = []
#         for sentence in sentences:
#             seq = [word_to_idx[word] for word in sentence.split() if word in word_to_idx]
#             seqs.append(torch.tensor(seq, dtype=torch.long))
#         return pad_sequence(seqs, batch_first=True, padding_value=0)

#     train_x_tensor = sentences_to_indices(training_data, word_to_idx)
#     val_x_tensor   = sentences_to_indices(validation_data, word_to_idx)

#     train_y_tensor = torch.tensor(training_labels, dtype=torch.long)
#     val_y_tensor   = torch.tensor(validation_labels, dtype=torch.long)

#     return train_x_tensor, train_y_tensor, val_x_tensor, val_y_tensor, vocab_size





    #     word_vectorizer = TfidfVectorizer( # Text till siffror
    #     analyzer='word',
    #     ngram_range=(1,2), # rangen är mellan ett ord och tvåpar
    #     max_features=50000,
    #     max_df=0.5, # Tar bort ord som finns i över 50% av alla texter, i engelskan är det exempelvis orden: This, The, Is, a och and.
    #     use_idf=True,
    #     norm='l2' # Normaliserar vektorerna
    # )
    # training_data = word_vectorizer.fit_transform(training_data).todense()
    # validation_data = word_vectorizer.transform(validation_data).todense()
    # train_x = torch.from_numpy(np.array(training_data)).float() # Gör från NumPy array till PyTorch Tensor. Vi sparar i float [2.0, 0,3. 0,6 etc]
    # train_y = torch.from_numpy(np.array(training_labels)).long() # Vi sparar i long [0, 1, 0 etc]
    # val_x = torch.from_numpy(np.array(validation_data)).float()
    # val_y = torch.from_numpy(np.array(validation_labels)).long()
    # return train_x, train_y, val_x, val_y

<>:23: SyntaxWarning: invalid escape sequence '\.'
<>:23: SyntaxWarning: invalid escape sequence '\.'
C:\Users\david\AppData\Local\Temp\ipykernel_9092\1639201171.py:23: SyntaxWarning: invalid escape sequence '\.'
  data['Sentence'] = data['Sentence'].replace('((25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)(\.|$)){4}', '', regex=True)    # remove IP address
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## imports

In [45]:
import comet_ml
from comet_ml import start
from comet_ml.integration.pytorch import log_model

import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, DataLoader
from torch.utils.data import random_split

from importlib import reload
import pandas as pd
from transformers import DistilBertTokenizer, DistilBertModel

from transformers import GPT2Model

#from importlib import reload
#import data_loading_code
#reload(data_loading_code)
#from data_loading_code import *


## Load Data

Might be wise to not load the 25K file if it isn't needed then only load the regular one.

In [15]:
train_dataset_LSTM = TensorDataset(train_x_tensor, train_y_tensor)
train_loader_LSTM = DataLoader(train_dataset_LSTM, batch_size=32, shuffle=True)

val_dataset_LSTM = TensorDataset(validation_x_tensor, validation_y_tensor)
val_loader_LSTM = DataLoader(val_dataset_LSTM, batch_size=32)

In [16]:
input_size = train_x_tensor.shape[1]
print(input_size)

class LSTM(nn.Module):
    def __init__(self, hidden_size = 128):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first= True)
        self.dropout = nn.Dropout(0.15)
        self.fc = nn.Linear(hidden_size, 2)

    def forward(self, x):
        x = x.unsqueeze(1)  # (batch, 1, features)
        out, (hidden, cell) = self.lstm(x)
        return self.fc(hidden[-1])

7277


# Task 1.1
A simple neural network composed of linear layers. You may incorporate activation
functions, dropout, and other complementary layers as needed.

In [ ]:
# settings
epochs = 5 #10
lr=0.0001

# scheduler settings
patience = 3
factor = 0.1

# initialize comet
lab1_1 = comet_ml.Experiment(
    api_key="wCXnRD5xewUGxCxBYe8ePt4JY",
    workspace="kanskejoanna",
    project_name="lab1",
    name="model 1.1.1 - ANN",
    display_name="model 1.1.1 - ANN",
)

# Report multiple hyperparameters using a dictionary:
hyper_params = {
   "learning_rate": lr,
   "steps": epochs
}
lab1_1.log_parameters(hyper_params)

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/kanskejoanna/lab1/b2e9acaf9b604563a5de7bc4a2239677



COMET WARNING: Unknown error exporting current conda environment
COMET WARNING: Unknown error retrieving Conda package as an explicit file
COMET WARNING: Unknown error retrieving Conda information


### 1.1 ANN: Running model

In [18]:
model_1_1 = nn.Sequential(
    nn.Linear(input_size, 128),
    nn.ReLU(),
    nn.Dropout(0.15),
    nn.Linear(128,2) # Negative or positive review
)

criterion = nn.CrossEntropyLoss()
#optimizer = torch.optim.SGD(model_LSTM.parameters(), lr=0.0001)
optimizer = torch.optim.Adam(model_1_1.parameters(), lr=lr)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor = factor,
        patience = patience # Wait x amount of epochs before reducing lr
    )

best_val_loss = float('inf')

for epoch in range(epochs):
    model_1_1.train()

    train_running_loss = 0.0
    for batch_x, batch_y in train_loader_LSTM:

#------------------------- TRAINING -------------------------
        optimizer.zero_grad()
        output = model_1_1(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        train_running_loss+=loss.item()
    average_train_loss = train_running_loss/len(train_loader_LSTM)
    lab1_1.log_metric("model 1.1.1 - train loss", average_train_loss, step=epoch)

#------------------------- VALIDATION -------------------------
    model_1_1.eval()
    val_running_loss = 0.0
    with torch.no_grad():
        for batch_x, batch_y in val_loader_LSTM:
            output = model_1_1(batch_x)
            val_loss = criterion(output, batch_y)
            val_running_loss += val_loss.item()
        average_val_loss = val_running_loss/len(val_loader_LSTM)
    lab1_1.log_metric("model 1.1.1 - validation loss", average_val_loss, step=epoch)

    scheduler.step(average_val_loss)
    lab1_1.log_metric("model 1.1.1 - learning rate", optimizer.param_groups[0]['lr'], step=epoch)

#------------------------- VISUALIZATION -------------------------
    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"train loss: {average_train_loss:.3f}, "
        f"val loss: {average_val_loss:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )
#------------------------- SAVING BEST MODEL -------------------------
    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        torch.save(model_1_1.state_dict(), "BestModel_1_1.pth")
        print("Saved The Best Performing Model")

lab1_1.end()

Epoch [1/5] train loss: 0.693, val loss: 0.692, lr: 0.000100
Saved The Best Performing Model
Epoch [2/5] train loss: 0.689, val loss: 0.690, lr: 0.000100
Saved The Best Performing Model
Epoch [3/5] train loss: 0.685, val loss: 0.686, lr: 0.000100
Saved The Best Performing Model


COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : dusty_wombat_1860
COMET INFO:     url                   : https://www.comet.com/kanskejoanna/lab1/b2e9acaf9b604563a5de7bc4a2239677
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     loss [14]                         : (0.6644008159637451, 0.6950333118438721)
COMET INFO:     model 1.1.1 - learning rate       : 0.0001
COMET INFO:     model 1.1.1 - train loss [5]      : (0.6693144748950827, 0.6926693094187769)
COMET INFO:     model 1.1.1 - validation loss [5] : (0.6759963780641556, 0.6918809860944748)
COMET INFO:   Parameters:
COMET INFO:     learning_rate : 0.0001
COMET INFO:     steps         : 5
COMET INFO:   Uploads:
COMET INFO:     environme

Epoch [4/5] train loss: 0.678, val loss: 0.682, lr: 0.000100
Saved The Best Performing Model
Epoch [5/5] train loss: 0.669, val loss: 0.676, lr: 0.000100
Saved The Best Performing Model


## 1.1 ANN: Test run
The best model is tested

In [19]:
#Test the model
model_1_1.load_state_dict(torch.load("BestModel_1_1.pth"))
model_1_1.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for batch_x, batch_y in val_loader_LSTM:
        output = model_1_1(batch_x)
        val_loss = criterion(output, batch_y)
        test_loss += val_loss.item()

        HighestValue, predicted = torch.max(output, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

average_test_loss = test_loss / len(val_loader_LSTM)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")

Test loss: 0.676
Test accuracy: 79.00%


C:\Users\david\AppData\Local\Temp\ipykernel_9092\2998430035.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_1_1.load_state_dict(torch.load("BestModel_1_1.pth"))


# Task 1.1 LSTM network
A neural network based on LSTM or bidirectional LSTM (Bi-LSTM) layers.

In [20]:
# settings
epochs = epochs #50
lr=0.0001

# scheduler settings
patience = 3
factor = 0.1

# initialize comet
lab1_1_2 = comet_ml.Experiment(
    api_key="wCXnRD5xewUGxCxBYe8ePt4JY",
    workspace="kanskejoanna",
    project_name="lab1",
    name="model 1.1.2 - Bi-LSTM",
)

# Report multiple hyperparameters using a dictionary:
hyper_params = {
   "learning_rate": lr,
   "steps": epochs
}
lab1_1_2.log_parameters(hyper_params)

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/kanskejoanna/lab1/3ecb4b7d3d1149c4a92ce00966948715

COMET WARNING: Unknown error exporting current conda environment
COMET WARNING: Unknown error retrieving Conda package as an explicit file
COMET WARNING: Unknown error retrieving Conda information


## 1.1 LSTM: Training and Validation
The model is trained, validated and the best model is saved.

In [21]:


model_LSTM = LSTM()
criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.SGD(model_LSTM.parameters(), lr=0.0001)
optimizer = torch.optim.Adam(model_LSTM.parameters(), lr=lr)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor = 0.1,
        patience = 3 # Wait x amount of epochs before reducing lr
    )

best_val_loss_LSTM = float('inf')


for epoch in range(epochs):
    model_LSTM.train()

    running_train_loss_LSTM = 0.0
    for batch_x, batch_y in train_loader_LSTM:
#------------------------- TRAINING -------------------------
        optimizer.zero_grad()
        output = model_LSTM(batch_x)
        loss_LSTM = criterion(output, batch_y)
        loss_LSTM.backward()
        optimizer.step()
        running_train_loss_LSTM += loss_LSTM.item()
    average_train_loss_LSTM = running_train_loss_LSTM/len(train_loader_LSTM)
    lab1_1_2.log_metric("model 1.1.2 - train loss", average_train_loss, step=epoch)

#------------------------- VALIDATION -------------------------
    model_LSTM.eval()
    val_running_loss_LSTM = 0.0
    with torch.no_grad():
        for batch_x, batch_y in val_loader_LSTM:
            output = model_LSTM(batch_x)
            val_loss_LSTM = criterion(output, batch_y)
            val_running_loss_LSTM += val_loss_LSTM.item()
        average_val_loss_LSTM = val_running_loss_LSTM/len(val_loader_LSTM)
        lab1_1_2.log_metric("model 1.1.2 - validation loss", average_val_loss_LSTM, step=epoch)

    scheduler.step(average_val_loss_LSTM)
    lab1_1_2.log_metric("model 1.1.2 - learning rate", optimizer.param_groups[0]['lr'], step=epoch)

#------------------------- VISUALIZATION -------------------------
    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"train loss: {average_train_loss_LSTM:.3f}, "
        f"val loss: {average_val_loss_LSTM:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )
#------------------------- SAVING BEST MODEL -------------------------
    if average_val_loss_LSTM < best_val_loss_LSTM:
        best_val_loss_LSTM = average_val_loss_LSTM
        torch.save(model_LSTM.state_dict(), "BestModel_LSTM.pth")
        print("Saved The Best Performing Model")




lab1_1_2.end()

Epoch [1/5] train loss: 0.697, val loss: 0.691, lr: 0.000100
Saved The Best Performing Model
Epoch [2/5] train loss: 0.697, val loss: 0.690, lr: 0.000100
Saved The Best Performing Model
Epoch [3/5] train loss: 0.692, val loss: 0.689, lr: 0.000100
Saved The Best Performing Model
Epoch [4/5] train loss: 0.691, val loss: 0.688, lr: 0.000100
Saved The Best Performing Model


COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : disappointed_template_9570
COMET INFO:     url                   : https://www.comet.com/kanskejoanna/lab1/3ecb4b7d3d1149c4a92ce00966948715
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     loss [14]                         : (0.6631344556808472, 0.713355302810669)
COMET INFO:     model 1.1.2 - learning rate       : 0.0001
COMET INFO:     model 1.1.2 - train loss          : 0.6693144748950827
COMET INFO:     model 1.1.2 - validation loss [5] : (0.6859978437423706, 0.6911266446113586)
COMET INFO:   Parameters:
COMET INFO:     learning_rate : 0.0001
COMET INFO:     steps         : 5
COMET INFO:   Uploads:
COMET INFO:     environment details    

Epoch [5/5] train loss: 0.686, val loss: 0.686, lr: 0.000100
Saved The Best Performing Model


## 1.1 LSTM Test run
The best model is tested

In [22]:
model_LSTM.load_state_dict(torch.load("BestModel_LSTM.pth"))
model_LSTM.eval()

test_loss_LSTM = 0.0
correct = 0
total = 0

with torch.no_grad():
    for batch_x, batch_y in val_loader_LSTM:
        output = model_LSTM(batch_x)
        val_loss_LSTM = criterion(output, batch_y)
        test_loss_LSTM += val_loss_LSTM.item()

        HighestValue, predicted = torch.max(output, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

average_test_loss_LSTM = test_loss_LSTM / len(val_loader_LSTM)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss_LSTM:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")



Test loss: 0.686
Test accuracy: 53.00%


C:\Users\david\AppData\Local\Temp\ipykernel_9092\3094863469.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_LSTM.load_state_dict(torch.load("BestModel_LSTM.pth"))


# Task 1.2: Transformers Implementation
For this task, you will implement your transformer in PyTorch. 

In [23]:
# settings
epochs = epochs #50
best_val_loss_bert = float('inf')
dataset = "amazon_cells_labelled_LARGE_25K.txt"
#lr = 0.0001

# scheduler settings
patience = 5
factor = 0.1
patience_counter = 0

# initialize comet
lab1_2 = comet_ml.Experiment(
    api_key="wCXnRD5xewUGxCxBYe8ePt4JY",
    workspace="kanskejoanna",
    project_name="lab1",
    display_name="model 1.1.2 - Bi-LSTM",
)

# Report multiple hyperparameters using a dictionary:
hyper_params = {
   "learning_rate": lr,
   "steps": epochs
}
lab1_2.log_parameters(hyper_params)

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/kanskejoanna/lab1/3f15e6a219564a37b120c699228546af

COMET WARNING: Unknown error exporting current conda environment
COMET WARNING: Unknown error retrieving Conda package as an explicit file
COMET WARNING: Unknown error retrieving Conda information


In [24]:
df = pd.read_csv(dataset, delimiter='\t', header=None)
sentiment_mapping = {0: 'Negative', 1: 'Positive'}
df['Rating'] = df[1].map(sentiment_mapping)#df['Rating'] = df[1].map(sentiment_mapping)
print('Head: \n')
print(df.head())

print('Rating: \n')
print(df['Rating'].value_counts())

Head: 

                                                   0  1    Rating
0  I've read this book with much expectation, it ...  0  Negative
1  love it...very touch.it's to bad that in the d...  1  Positive
2  The creepiest book I've ever read! It's a cree...  1  Positive
3  It starts off a bit slow, but once the product...  1  Positive
4  As good as this book may be, the print quality...  0  Negative
Rating: 

Rating
Positive    15116
Negative     9884
Name: count, dtype: int64


In [25]:
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, csv_file, tokenizer, max_length):
        self.dataset = pd.read_csv(csv_file, delimiter='\t', header=None, names=['Sentence', 'Class'])
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.label_dict = {0: 'Negative', 1: 'Positive'}
    

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        review = self.dataset.iloc[idx, 0]
        sentiment = self.dataset.iloc[idx, 1]
        label = self.label_dict[sentiment]

        encoding = self.tokenizer(
            review,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt',
            truncation=True
        )

        return {
            'review_text': review,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(sentiment, dtype=torch.long)
        }

In [26]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
review_dataset = ReviewDataset(dataset, tokenizer, 512)
review_dataset[0]
tokenizer.decode(review_dataset[0]['input_ids'])

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

c:\Users\david\anaconda3\envs\tretolv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\david\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

"[CLS] i ' ve read this book with much expectation, it was very boring all through out the book [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD

In [27]:
train_size_bert = int(0.7 * len(df))
val_size_bert = int(0.15 * len(df))
test_size_bert = int(0.15 * len(df))
train_dataset_bert, val_dataset_bert, test_dataset_bert = random_split(review_dataset, [train_size_bert, val_size_bert, test_size_bert])

train_loader_bert = DataLoader(train_dataset_bert, batch_size=16, shuffle=True)
val_loader_bert = DataLoader(val_dataset_bert, batch_size=16, shuffle=False)
test_loader_bert = DataLoader(test_dataset_bert, batch_size=16, shuffle=False)

print(f"Number of training samples: {len(train_dataset_bert)}")
print(f"Number of validation samples: {len(val_dataset_bert)}")
print(f"Number of test samples: {len(test_dataset_bert)}")

Number of training samples: 17500
Number of validation samples: 3750
Number of test samples: 3750


In [28]:
class CustomDistilBertForClassification(nn.Module):
    def __init__(self, num_labels=2):
        super(CustomDistilBertForClassification, self).__init__()
        self.distilbert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.pre_classifier = nn.Linear(self.distilbert.config.dim, 64)
        self.dropout = nn.Dropout(0.4)
        self.classifier = nn.Linear(64, num_labels)

    def forward(self, input_ids, attention_mask):
        distilbert_output = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = distilbert_output[0]  # (batch_size, sequence_length, hidden_size)
        pooled_output = hidden_state[:, 0]  # Take the [CLS] token representation
        pooled_output = self.dropout(pooled_output)
        pooled_output = self.pre_classifier(pooled_output)
        pooled_output = nn.ReLU()(pooled_output)
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

In [29]:
model = CustomDistilBertForClassification()

print(model.distilbert)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

## 1.2 Transformer: Training and Validation

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
# Add weight decay (L2 regularization) to reduce overfitting
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=0.05)

# Learning rate scheduler to reduce LR when validation loss plateaus
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=patience,
    #verbose=True,
    min_lr=1e-7
)



model.train()
for epoch in range(epochs):
    # Training phase
    total_loss = 0
    for batch in train_loader_bert:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader_bert)
    lab1_1_2.log_metric("model 1.2 - train loss", average_train_loss, step=epoch)
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader_bert:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(val_loader_bert)
    lab1_2.log_metric("model 1.2 - validation loss", avg_val_loss, step=epoch)

    model.train()
    
    # Step scheduler
    scheduler.step(avg_val_loss)
    lab1_2.log_metric("model 1.2 - learning rate", optimizer.param_groups[0]['lr'], step=epoch)
    
    # Print metrics
    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
    
    # Save best model and implement early stopping
    if avg_val_loss < best_val_loss_bert:
        best_val_loss_bert = avg_val_loss
        torch.save(model.state_dict(), "BestModel_DistilBert.pth")
        print(f"Saved best model with validation loss: {avg_val_loss:.4f}")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            break

lab1_2.end()

Epoch 1/5, Train Loss: 0.3702, Val Loss: 0.3312
Saved best model with validation loss: 0.3312
Epoch 2/5, Train Loss: 0.3424, Val Loss: 0.3113
Saved best model with validation loss: 0.3113
Epoch 3/5, Train Loss: 0.3099, Val Loss: 0.3130
Epoch 4/5, Train Loss: 0.2762, Val Loss: 0.3124


COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : medical_sailfish_8311
COMET INFO:     url                   : https://www.comet.com/kanskejoanna/lab1/3f15e6a219564a37b120c699228546af
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     loss [546]                      : (0.039872217923402786, 0.965618908405304)
COMET INFO:     model 1.2 - learning rate       : 5e-05
COMET INFO:     model 1.2 - validation loss [5] : (0.31125216450779997, 0.3311910338224249)
COMET INFO:   Parameters:
COMET INFO:     learning_rate : 0.0001
COMET INFO:     steps         : 5
COMET INFO:   Uploads:
COMET INFO:     environment details      : 1
COMET INFO:     filename                 : 1
COMET INFO:     git metadata 

Epoch 5/5, Train Loss: 0.2509, Val Loss: 0.3128


COMET INFO: Please wait for metadata to finish uploading (timeout is 3600 seconds)
COMET INFO: Uploading 25 metrics, params and output messages


## 1.2 Transformer: Testing

In [31]:

model.load_state_dict(torch.load("BestModel_DistilBert.pth"))
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader_bert:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)
        test_loss += loss.item()
        
        # Calculate accuracy
        _, predicted = torch.max(logits, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

avg_test_loss = test_loss / len(val_loader_bert)
test_accuracy = 100 * (correct / total)

print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")
print(f"Correct Predictions: {correct}/{total}")

C:\Users\david\AppData\Local\Temp\ipykernel_9092\116359246.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("BestModel_DistilBert.pth"))


Test Loss: 0.3016
Test Accuracy: 88.13%
Correct Predictions: 3305/3750


## 1.2 GPT 2

In [53]:
class CustomGPT2ForSequenceClassification(nn.Module):
    def __init__(self, num_labels=2):
        super().__init__()

        self.gpt2 = GPT2Model.from_pretrained("gpt2")
        hidden_size = self.gpt2.config.hidden_size

        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(hidden_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.classifier = nn.Linear(64, num_labels)
        self.relu = nn.ReLU()

    def forward(self, input_ids, attention_mask):
        outputs = self.gpt2(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state

        last_token_idx = attention_mask.sum(dim=1) - 1
        batch_size = input_ids.size(0)

        pooled_output = hidden_states[
            torch.arange(batch_size, device=input_ids.device),
            last_token_idx
        ]

        x = self.dropout(pooled_output)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.fc2(x)
        x = self.relu(x)
        x = self.dropout(x)

        logits = self.classifier(x)
        return logits

In [56]:

from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

#model = GPT2ForSequenceClassification.from_pretrained("gpt2", num_labels=2)
model = CustomGPT2ForSequenceClassification(num_labels=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

cuda


In [64]:
csv_file_path = "amazon_cells_labelled_LARGE_25K.txt"
dataset = ReviewDataset(csv_file_path, tokenizer, max_length=128)

print(dataset[0])
print(tokenizer.decode(dataset[0]["input_ids"], skip_special_tokens=True))

{'review_text': "I've read this book with much expectation, it was very boring all through out the book", 'input_ids': tensor([   40,  1053,  1100,   428,  1492,   351,   881, 17507,    11,   340,
          373,   845, 14262,   477,   832,   503,   262,  1492, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 5

In [58]:
train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size]
)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

print(f"Train: {len(train_dataset)}")
print(f"Val:   {len(val_dataset)}")
print(f"Test:  {len(test_dataset)}")

Train: 700
Val:   150
Test:  150


In [59]:
gpt2_logger = comet_ml.Experiment(
    api_key="wCXnRD5xewUGxCxBYe8ePt4JY",
    workspace="kanskejoanna",
    project_name="lab1",
    display_name="GPT2 Model",
)

hyper_params = {
   "learning_rate": lr,
   "steps": epochs
}
gpt2_logger.log_parameters(hyper_params)

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/kanskejoanna/lab1/73cc65c170724c35bc2c4f9177b43f05

COMET WARNING: Unknown error exporting current conda environment
COMET WARNING: Unknown error retrieving Conda package as an explicit file
COMET WARNING: Unknown error retrieving Conda information


In [ ]:
criterion = nn.CrossEntropyLoss()
best_val_loss = float("inf")
patience_counter = 0
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=0.05)

# Learning rate scheduler to reduce LR when validation loss plateaus
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=patience,
    min_lr=1e-7
)

for epoch in range(epochs):
    # Training phase
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader)
    gpt2_logger.log_metric("GPT2 - train loss", avg_train_loss, step=epoch)
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(val_loader)
    gpt2_logger.log_metric("GPT2 - validation loss", avg_val_loss, step=epoch)

    model.train()
    
    # Step scheduler
    scheduler.step(avg_val_loss)
    gpt2_logger.log_metric("GPT2 - Learning Rate", optimizer.param_groups[0]['lr'], step=epoch)
    
    # Print metrics
    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
    
    # Save best model and implement early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "BestModel_GPT2.pth")
        print(f"Saved best model with validation loss: {avg_val_loss:.4f}")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            break

gpt2_logger.end()

Epoch 1/5, Train Loss: 0.7216, Val Loss: 0.6723
Saved best model with validation loss: 0.6723
Epoch 2/5, Train Loss: 0.6655, Val Loss: 0.5326
Saved best model with validation loss: 0.5326
Epoch 3/5, Train Loss: 0.4930, Val Loss: 0.3014
Saved best model with validation loss: 0.3014
Epoch 4/5, Train Loss: 0.3608, Val Loss: 0.2131
Saved best model with validation loss: 0.2131
Epoch 5/5, Train Loss: 0.5461, Val Loss: 0.2850


### Test GPT2

In [63]:

model.load_state_dict(torch.load("BestModel_GPT2.pth"))
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)
        test_loss += loss.item()
        
        # Calculate accuracy
        _, predicted = torch.max(logits, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

avg_test_loss = test_loss / len(val_loader_bert)
test_accuracy = 100 * (correct / total)

print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")
print(f"Correct Predictions: {correct}/{total}")

C:\Users\david\AppData\Local\Temp\ipykernel_9092\3752993064.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("BestModel_GPT2.pth"))


Test Loss: 0.0138
Test Accuracy: 94.67%
Correct Predictions: 142/150


## Task 1.3 Comparison
Here, you should compare of all three models; you are requested to use the same test dataset
for Simple ANN, LSTM based model and the Transformer to answer the following:

• Compare the performance of the two models and explain in which scenarios you would
prefer one over the other.

• How did the two models’ complexity, accuracy, and efficiency differ? Did one model
outperform the other in specific scenarios or tasks? If so, why?

• What insights did you obtain concerning data amount to train? Embedding utilized?
Architectural choices made?